# BRCA Metabolic Flux Pipeline — 02. Supervised, unsupervised and clinical analysis

Reads the outputs of notebook 01 (`results/pythonResults/processedData`) and of `matlab/BRCA_fluxAnalysis.m`
(`results/matlabResults`) and produces every number, table and figure of the manuscript (Fig 1 is the workflow schematic)
(`results/pythonResults/tables`, `results/pythonResults/figures`; figures are also written with PLOS names to
`reportes/articulo/figures`).

| Section | Manuscript |
|---|---|
| 1 Model content and variability | Fig 2 |
| 2 Tumor vs normal pathways | Fig 3, S5 Table (reactions and genes) |
| 3 Supervised learning (nested, patient-grouped CV; held-out test) | Tables 2-3, Figs 4-7 |
| 4 Flux solutions compared | S1 Fig (reviewer 1, comment 1) |
| 5 Unsupervised clustering, clinical concordance | Table 4, Figs 8-10 |
| 6 Characterization of metabolic clusters | Fig 11, S6 Table |
| 7 Clinical variables encoded in fluxes | Fig 12 |
| 8 Metabolic signatures across PAM50 subtypes | Fig 13, S7 Table |
| 9 Survival (supplementary) and Pareto surfaces | S2 Fig, S3 Fig |

Design: models failing MATLAB QC are excluded; tumor and normal samples of one patient are always in the
same fold; feature selection and hyper-parameter search happen inside training folds only; clusters are tested
against technical covariates (missing metadata, tissue source site, medium supplementation, model size) before being interpreted.
Only numpy/scipy/scikit-learn/matplotlib are needed (UMAP and survival statistics are in `brca_analysis.py`).

In [ ]:
import os, re, json, time, datetime, warnings, sys
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

DRIVE_DIR  = Path(os.environ.get("BRCA_DRIVE_DIR", Path.home() / "work/Drive/GIMI/01_posgrado/MCBCI_analisisCancer"))
PROC_DIR   = DRIVE_DIR / "results" / "pythonResults" / "processedData"
MATLAB_DIR = DRIVE_DIR / "results" / "matlabResults"
OUT_DIR    = DRIVE_DIR / "results" / "pythonResults"
REF_DIR    = DRIVE_DIR / "data" / "reference"
PAPER_FIG_DIR = DRIVE_DIR / "reportes" / "articulo" / "figures"

SOLS   = ["FBA", "pFBA", "L1w"]          # flux solutions used in the main text
SEEDS  = [42, 123, 100]
N_ITER = 6                               # random-search iterations per classifier (inner CV)
N_JOBS = -1
FAST   = False
# >>> CONFIG END

In [ ]:
sys.path.insert(0, str(Path.cwd()))
import brca_analysis as ba
from brca_analysis import COL, ALGO_COLORS, CLUSTER_COLORS, bh, cramers_v, cliffs_matrix
warnings.filterwarnings("ignore")
ba.set_style()
if FAST:
    SEEDS, N_ITER = SEEDS[:1], 2
FIG, TAB = OUT_DIR / "figures", OUT_DIR / "tables"
for d in (FIG, TAB, PAPER_FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
NUM = {}                                 # numbers quoted in the manuscript

def savefig(fig, name, paper_name=None):
    ba.save(fig, str(FIG / name))
    if paper_name:
        import shutil
        shutil.copy(FIG / f"{name}.tif", PAPER_FIG_DIR / f"{paper_name}.tif")
        shutil.copy(FIG / f"{name}.png", PAPER_FIG_DIR / f"{paper_name}.png")
    print("saved", name)

# Fig 1 (workflow schematic, vector)
ba.workflow_figure(str(FIG / "Fig1_workflow"))
import shutil as _sh
for _e in ("tif", "png"):
    _sh.copy(FIG / f"Fig1_workflow.{_e}", PAPER_FIG_DIR / f"Fig1.{_e}")

## 0. Data

In [ ]:
def to_sample_id(s):
    m = re.search(r"TCGA[-_.][A-Z0-9]{2}[-_.][A-Z0-9]{4}[-_.]\d{2}[A-Z]?", str(s))
    return re.sub(r"[_.]", "-", m.group(0))[:16] if m else str(s)

feat = pd.read_csv(MATLAB_DIR / "FeatureMatrix_TumorPhenotype_All.csv", low_memory=False)
feat["sample"] = feat["Model"].map(to_sample_id)
qc   = pd.read_csv(MATLAB_DIR / "ModelQC.csv"); qc["sample"] = qc["Model"].map(to_sample_id)
inv  = pd.read_csv(PROC_DIR / "sample_inventory.csv")
clin = pd.read_csv(PROC_DIR / "clinical_processed.csv")
rxn_ref = pd.read_csv(REF_DIR / "Recon3D_reactions.csv")
NUM["n_models"] = int(len(feat)); NUM["n_qc_pass"] = int(feat.QC_pass.sum())
NUM["qc_status"] = qc.Status.value_counts().to_dict()
print(NUM)

df = (feat[feat.QC_pass == 1]
      .merge(inv[["sample", "patient", "sample_type", "tss_code", "is_first_tumor_vial", "paired_tumor_normal", "in_metadata"]], on="sample")
      .merge(clin.drop(columns=["patient", "sample_type", "tss_code", "tss_name", "in_metadata"]), on="sample", how="left")
      .reset_index(drop=True))
NUM["n_by_type"] = df.sample_type.value_counts().to_dict()

cols = df.columns
def metric_cols(sols):
    return [c for c in cols if c.rsplit("_", 1)[-1] in sols and not c.startswith("Flux_") and c not in ("MaxBiomass_FBA",)]
def flux_cols(sols):
    return [c for c in cols if c.startswith("Flux_") and c.rsplit("_", 1)[-1] in sols]
def clean(X, min_nonzero=0.05):
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X[X.columns[(X.fillna(0) != 0).mean() >= min_nonzero]]
    return X[X.columns[X.std() > 1e-12]]

FD = pd.read_csv(MATLAB_DIR / "FeatureDictionary.csv")
flux2rxn = dict(zip(FD.Column, FD.Recon3D_ID_or_Name))
ss_of = dict(zip(rxn_ref.rxns, rxn_ref.subSystems))
gpr_of = dict(zip(rxn_ref.rxns, rxn_ref.grRules))
name_of = dict(zip(rxn_ref.rxns, rxn_ref.rxnNames))

## 1. Structural characterization of the patient-specific models (Fig 2)

In [ ]:
pm_dir = MATLAB_DIR / "perModel_uniformMedium"
if not pm_dir.exists():
    pm_dir = MATLAB_DIR / "perModel"
rsets = {}
for s in df["sample"]:
    f = pm_dir / f"{s}.mat"
    if f.exists():
        R = sio.loadmat(f, squeeze_me=True, struct_as_record=False, variable_names=["R"])["R"]
        rsets[s] = set(map(str, np.atleast_1d(R.rxns)))
allr = sorted(set().union(*rsets.values()))
ridx = {r: i for i, r in enumerate(allr)}
samples = [s for s in df["sample"] if s in rsets]
B = np.zeros((len(samples), len(allr)), dtype=np.float32)
for i, s in enumerate(samples):
    B[i, [ridx[r] for r in rsets[s]]] = 1
inter = B @ B.T
sizes = B.sum(1)
J = inter / (sizes[:, None] + sizes[None, :] - inter)
np.fill_diagonal(J, np.nan)
meanJ = np.nanmean(J, axis=1)
freq = B.mean(0)
size_tab = df.set_index("sample").loc[samples, ["sample_type", "nRxns", "nMets", "nGenes"]].assign(mean_jaccard=meanJ)
size_tab.to_csv(TAB / "model_size_similarity.csv")
NUM.update({"rxns_mean": float(df.nRxns.mean()), "rxns_sd": float(df.nRxns.std()), "mets_mean": float(df.nMets.mean()),
            "genes_mean": float(df.nGenes.mean()), "rxns_union": len(allr), "rxns_core_all": int((freq == 1).sum()),
            "rxns_core_95": int((freq >= 0.95).sum()), "jaccard_mean": float(np.nanmean(J)), "jaccard_sd": float(np.nanstd(J)),
            "rxns_lt5pct": int((freq < 0.05).sum())})
print({k: NUM[k] for k in ["rxns_mean", "rxns_union", "rxns_core_all", "jaccard_mean"]})
# Jaccard on internal reactions only (no exchange/demand/sink/transport) and within/between tissue types
_bnd = np.array([r.startswith(("EX_", "DM_", "sink_")) or str(ss_of.get(r, "")).startswith(("Transport", "Exchange")) for r in allr])
Bi = B[:, ~_bnd]; ii = Bi @ Bi.T; si = Bi.sum(1)
Ji = ii / (si[:, None] + si[None, :] - ii); np.fill_diagonal(Ji, np.nan)
_ty = size_tab.sample_type.values
_jt = {}
for a_, b_ in [("Tumor", "Tumor"), ("Normal", "Normal"), ("Tumor", "Normal")]:
    blk = J[np.ix_(_ty == a_, _ty == b_)]
    _jt[f"{a_}-{b_}"] = float(np.nanmean(blk))
NUM.update({"rxns_boundary": int(_bnd.sum()), "jaccard_internal_mean": float(np.nanmean(Ji)), "jaccard_by_tissue": _jt})
print(NUM["jaccard_internal_mean"], _jt)

fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.7), gridspec_kw={"width_ratios": [0.85, 1.15, 1]})
TYPES = [t for t in ["Tumor", "Normal", "Metastatic"] if (size_tab.sample_type == t).any()]
ax = axes[0]
bins = np.linspace(size_tab.nRxns.min(), size_tab.nRxns.max(), 30)
for t in ["Tumor", "Normal"]:
    m = size_tab.sample_type == t
    ax.hist(size_tab.nRxns[m], bins=bins, density=True, histtype="stepfilled", alpha=0.45, color=COL[t], label=t)
    ax.hist(size_tab.nRxns[m], bins=bins, density=True, histtype="step", lw=1.0, color=COL[t])
ax.set_xlabel("Reactions per model"); ax.set_ylabel("Density"); ax.set_yticks([])
ax.legend(loc="upper left", handlelength=1.0, handletextpad=0.3)
ba.panel_label(ax, "a")
ax = axes[1]
for t in TYPES:
    m = size_tab.sample_type == t
    ax.scatter(size_tab.nRxns[m], size_tab.mean_jaccard[m], s=6 if t == "Tumor" else 10, c=COL[t], alpha=0.7,
               edgecolors="white", linewidths=0.25, label=f"{t} (n={m.sum()})", zorder=3 if t != "Tumor" else 2)
ax.set_xlabel("Reactions per model"); ax.set_ylabel("Mean Jaccard similarity")
ax.legend(loc="lower right", handletextpad=0.1, borderpad=0.2, fontsize=6.5, markerscale=1.3)
ba.panel_label(ax, "b")
ax = axes[2]
ax.hist(freq * 100, bins=40, color=COL["ink2"], edgecolor="white", linewidth=0.4)
ax.set_yscale("log"); ax.set_xlabel("Models containing the reaction (%)"); ax.set_ylabel("Reactions")
ax.axvline(100, color=COL["ink2"], lw=0.8, ls=":")
ax.text(96, ax.get_ylim()[1] * 0.6, f"{NUM['rxns_core_all']} in\nall models", ha="right", va="top", fontsize=6.5, color=COL["ink2"])
ax.text(6, ax.get_ylim()[1] * 0.6, f"{NUM['rxns_lt5pct']:,} in\n<5% of models", ha="left", va="top", fontsize=6.5, color=COL["ink2"])
ba.panel_label(ax, "c")
fig.tight_layout(w_pad=1.2)
savefig(fig, "Fig2_model_size_similarity", "Fig2")

## 2. Tumor vs normal: reactions and pathways (Fig 3)

In [ ]:
lab = df[df.sample_type.isin(["Tumor", "Normal"])].reset_index(drop=True)
yT = (lab.sample_type == "Tumor").values
rows = []
for s in SOLS:
    Xf = clean(lab[flux_cols([s])])
    d, p = cliffs_matrix(Xf[yT].fillna(0).values, Xf[~yT].fillna(0).values)
    rows.append(pd.DataFrame({"feature": Xf.columns, "solution": s, "cliffs_delta": d, "p": p}))
rx_stats = pd.concat(rows)
rx_stats["FDR"] = bh(rx_stats.p.values)
rx_stats["rxn"] = rx_stats.feature.map(flux2rxn)
rx_stats["subsystem"] = rx_stats.rxn.map(ss_of).fillna("Unassigned")
rx_stats["direction"] = np.where(rx_stats.cliffs_delta > 0, "higher in tumor", "higher in normal")
sig = rx_stats[rx_stats.FDR < 0.05]
best = sig.sort_values("FDR").drop_duplicates("rxn")            # one row per reaction (best solution)
best = best.assign(name=best.rxn.map(name_of), genes_Entrez=best.rxn.map(gpr_of))
best.sort_values(["subsystem", "FDR"])[["subsystem", "rxn", "name", "solution", "cliffs_delta", "p", "FDR", "direction", "genes_Entrez"]] \
    .to_csv(TAB / "S_Table_tumor_vs_normal_reactions.csv", index=False)
NUM["tn_reactions_tested"] = int(rx_stats.rxn.nunique()); NUM["tn_reactions_sig"] = int(best.rxn.nunique())
NUM["tn_sig_higher_tumour"] = int((best.cliffs_delta > 0).sum())

# subsystem activity (mean |flux|) test, used for the x axis
sa = []
for s in SOLS:
    Xs = clean(lab[[c for c in lab.columns if c.startswith("SA_") and c.endswith("_" + s)]])
    d, p = cliffs_matrix(Xs[yT].fillna(0).values, Xs[~yT].fillna(0).values)
    sa.append(pd.DataFrame({"feature": Xs.columns, "solution": s, "delta": d, "p": p}))
sa = pd.concat(sa); sa["FDR"] = bh(sa.p.values)
sa["ss_key"] = sa.feature.str.replace(r"^SA_|_(FBA|pFBA|L1w)$", "", regex=True)
ss_key = lambda x: re.sub(r"[^A-Za-z0-9]+", "_", str(x)).strip("_")
agg = best.groupby("subsystem").agg(n_sig=("rxn", "size"), frac_tumour=("cliffs_delta", lambda v: (v > 0).mean())).reset_index()
agg["ss_key"] = agg.subsystem.map(ss_key)
agg = agg.merge(sa.sort_values("FDR").drop_duplicates("ss_key")[["ss_key", "FDR", "delta"]], on="ss_key", how="left")
agg = agg[agg.subsystem != "Unassigned"].sort_values("n_sig", ascending=False)
agg.to_csv(TAB / "tumor_vs_normal_subsystems.csv", index=False)
top = agg.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(6.2, 4.2))
x = -np.log10(top.FDR.clip(lower=1e-300).fillna(1))
colors = [COL["Tumor"] if f >= 0.5 else COL["Normal"] for f in top.frac_tumour]
size = lambda n: 12 + 260 * np.sqrt(np.asarray(n, float) / agg.n_sig.max())
ax.scatter(x, np.arange(len(top)), s=size(top.n_sig), c=colors, alpha=0.85, edgecolors="white", linewidths=0.6, zorder=3)
ax.set_yticks(np.arange(len(top))); ax.set_yticklabels(top.subsystem)
ax.set_xlabel("Tumor vs normal subsystem activity (mean |flux|), −log$_{10}$ FDR"); ax.set_ylim(-0.7, len(top) - 0.3)
ax.grid(axis="x", color=COL["grid"], lw=0.6); ax.set_axisbelow(True)
ref = [n for n in [5, 20, int(agg.n_sig.max())] if n <= agg.n_sig.max()]
hs = [ax.scatter([], [], s=size(n), c=COL["muted"], label=f"{n}") for n in ref]
leg1 = ax.legend(handles=hs, title="Significant\nreactions", loc="upper left", bbox_to_anchor=(1.01, 1.0), labelspacing=1.3,
                 borderpad=0.6, fontsize=7, title_fontsize=7)
ax.add_artist(leg1)
ax.legend(handles=[Patch(color=COL["Tumor"], label="mostly higher\nin tumor"), Patch(color=COL["Normal"], label="mostly higher\nin normal")],
          loc="lower left", bbox_to_anchor=(1.01, 0.0), fontsize=7)
ax.set_xlim(0, x.max() * 1.1 + 0.5)
fig.tight_layout()
savefig(fig, "Fig3_tumor_normal_pathways", "Fig3")
print(top[["subsystem", "n_sig", "FDR", "frac_tumour"]].iloc[::-1].to_string())

## 3. Supervised learning: tumor vs normal (Tables 2-3, Figs 4-7)
Features: reaction fluxes and derived metrics of FBA, pFBA and L1w. Pipeline: median imputation → Mann-Whitney/BH
selection → standardization → classifier, all fitted on training folds. Outer loop: 5-fold StratifiedGroupKFold by
patient × 3 seeds; inner loop: randomized search (3-fold, grouped, ROC-AUC). Independent test: grouped 70/30 split.

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, matthews_corrcoef, confusion_matrix, roc_curve)
from scipy.stats import loguniform, randint, uniform

Xsup_df = clean(lab[flux_cols(SOLS) + metric_cols(SOLS)])
Xsup, ysup, gsup = Xsup_df.values.astype(float), yT.astype(int), lab.patient.values
NUM["sup_n"] = int(len(ysup)); NUM["sup_tumour"] = int(ysup.sum()); NUM["sup_normal"] = int((1 - ysup).sum())
NUM["sup_patients"] = int(pd.Series(gsup).nunique()); NUM["sup_features"] = int(Xsup.shape[1])
print(NUM["sup_n"], NUM["sup_features"])

SPACES = {
    "Logistic Regression": (LogisticRegression(max_iter=4000, class_weight="balanced"), {"clf__C": loguniform(1e-3, 10)}),
    "SVM": (SVC(probability=True, class_weight="balanced"), {"clf__C": loguniform(1e-2, 100), "clf__kernel": ["rbf", "linear"], "clf__gamma": ["scale", 1e-3, 1e-2]}),
    "KNN": (KNeighborsClassifier(), {"clf__n_neighbors": randint(3, 31), "clf__weights": ["uniform", "distance"], "clf__metric": ["euclidean", "manhattan", "cosine"]}),
    "Random Forest": (RandomForestClassifier(class_weight="balanced", n_jobs=1), {"clf__n_estimators": randint(100, 400), "clf__max_depth": randint(3, 16), "clf__max_features": ["sqrt", "log2"], "clf__min_samples_leaf": randint(1, 10)}),
    "Gradient Boosting": (HistGradientBoostingClassifier(class_weight="balanced"), {"clf__learning_rate": loguniform(0.02, 0.3), "clf__max_depth": randint(2, 8), "clf__max_iter": randint(100, 400), "clf__l2_regularization": loguniform(1e-4, 1)}),
    "MLP": (MLPClassifier(max_iter=600, early_stopping=True), {"clf__hidden_layer_sizes": [(64,), (128,), (64, 32)], "clf__alpha": loguniform(1e-5, 1e-1), "clf__learning_rate_init": loguniform(1e-4, 1e-2)}),
    "Decision Tree": (DecisionTreeClassifier(class_weight="balanced"), {"clf__max_depth": randint(2, 12), "clf__min_samples_leaf": randint(1, 20), "clf__criterion": ["gini", "entropy"]}),
    "Naive Bayes": (GaussianNB(), {"clf__var_smoothing": loguniform(1e-12, 1e-6)}),
    "Dummy (prior)": (DummyClassifier(strategy="prior"), {}),
}
ALGOS = list(SPACES)

def make_pipe(clf):
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("select", ba.MWUSelector(alpha=0.05, k_min=10)),
                     ("scale", StandardScaler()), ("clf", clf)])

def fit_model(name, Xtr, ytr, gtr, seed):
    clf, space = SPACES[name]
    pipe = make_pipe(clone(clf))
    if "random_state" in pipe.named_steps["clf"].get_params():
        pipe.set_params(clf__random_state=seed)
    if not space:
        return pipe.fit(Xtr, ytr), {}
    inner = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=seed)
    rs = RandomizedSearchCV(pipe, space, n_iter=N_ITER, cv=inner, scoring="roc_auc", random_state=seed, n_jobs=N_JOBS, error_score=np.nan)
    rs.fit(Xtr, ytr, groups=gtr)
    return rs.best_estimator_, rs.best_params_

def scores(y, prob):
    pred = (prob >= 0.5).astype(int)
    return {"Accuracy": accuracy_score(y, pred), "Balanced accuracy": balanced_accuracy_score(y, pred),
            "Precision": precision_score(y, pred, zero_division=0), "Sensitivity": recall_score(y, pred),
            "Specificity": recall_score(y, pred, pos_label=0), "F1-score": f1_score(y, pred), "MCC": matthews_corrcoef(y, pred),
            "ROC-AUC": roc_auc_score(y, prob)}

rows, oof, best_params = [], [], []
t0 = time.time()
for seed in SEEDS:
    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(outer.split(Xsup, ysup, gsup)):
        for name in ALGOS:
            t1 = time.time()
            model, bp = fit_model(name, Xsup[tr], ysup[tr], gsup[tr], seed)
            fit_s = time.time() - t1
            p_te = model.predict_proba(Xsup[te])[:, 1]; p_tr = model.predict_proba(Xsup[tr])[:, 1]
            r = {"model": name, "seed": seed, "fold": fold, "fit_seconds": fit_s,
                 "n_features": int(model.named_steps["select"].support_.sum()), "train_ROC-AUC": roc_auc_score(ysup[tr], p_tr)}
            r.update(scores(ysup[te], p_te))
            rows.append(r)
            oof.append(pd.DataFrame({"model": name, "seed": seed, "idx": te, "y": ysup[te], "prob": p_te}))
            best_params.append({"model": name, "seed": seed, "fold": fold, **{k.replace("clf__", ""): str(v) for k, v in bp.items()}})
    print(f"seed {seed} done ({(time.time() - t0) / 60:.1f} min)")
cv = pd.DataFrame(rows); oof = pd.concat(oof)
cv.to_csv(TAB / "supervised_nestedCV_folds.csv", index=False)
oof.to_csv(TAB / "supervised_nestedCV_predictions.csv", index=False)
# technical baseline: classifier using only reconstruction/QC variables (model size, active reactions, supplementation)
_tech = pd.DataFrame({"nRxns": lab.nRxns.values, "nMets": lab.nMets.values, "nGenes": lab.nGenes.values})
for s_ in SOLS:
    _fc = flux_cols([s_]); _tech[f"active_{s_}"] = (lab[_fc].fillna(0).abs() > 1e-9).sum(1).values
_tech["supplemented"] = lab["sample"].isin(set(qc.loc[qc.MediumSupplement.fillna("").astype(str).str.strip().ne(""), "sample"]) if "MediumSupplement" in qc else set()).astype(float).values
_auc_t = []
for seed in SEEDS:
    for tr, te in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed).split(_tech, ysup, gsup):
        for nm_, clf_ in [("Logistic Regression", LogisticRegression(max_iter=4000, class_weight="balanced")),
                          ("Gradient Boosting", HistGradientBoostingClassifier(class_weight="balanced", random_state=seed))]:
            pp_ = Pipeline([("s", StandardScaler()), ("c", clf_)]).fit(_tech.values[tr], ysup[tr])
            _auc_t.append({"model": nm_, "AUC": roc_auc_score(ysup[te], pp_.predict_proba(_tech.values[te])[:, 1])})
_auc_t = pd.DataFrame(_auc_t).groupby("model").AUC.agg(["mean", "std"]).round(3)
_auc_t.to_csv(TAB / "supervised_technical_baseline.csv")
NUM["technical_baseline_auc"] = _auc_t["mean"].to_dict(); print(_auc_t)

# sensitivity: models that needed medium supplementation (MATLAB rescueMedium) excluded from evaluation
_sup_ids = set(qc.loc[qc.MediumSupplement.fillna("").astype(str).str.strip().ne(""), "sample"]) if "MediumSupplement" in qc else set()
_is_sup = lab["sample"].isin(_sup_ids).values
NUM["sup_supplemented"] = {"Tumor": int((_is_sup & (ysup == 1)).sum()), "Normal": int((_is_sup & (ysup == 0)).sum())}
_sens = []
for (m_, s_), g_ in oof.groupby(["model", "seed"]):
    k_ = ~_is_sup[g_.idx.values]
    _sens.append({"model": m_, "seed": s_, "AUC_all": roc_auc_score(g_.y, g_.prob), "AUC_excl_supplemented": roc_auc_score(g_.y[k_], g_.prob[k_]),
                  "BA_all": balanced_accuracy_score(g_.y, g_.prob >= 0.5), "BA_excl_supplemented": balanced_accuracy_score(g_.y[k_], g_.prob[k_] >= 0.5)})
_sens = pd.DataFrame(_sens).groupby("model").mean(numeric_only=True).drop(columns="seed").round(3)
_sens.to_csv(TAB / "supervised_sensitivity_medium_supplement.csv")
NUM["sens_auc_maxdiff"] = float((_sens.AUC_excl_supplemented - _sens.AUC_all).abs().max())
print(NUM["sup_supplemented"], _sens)
pd.DataFrame(best_params).to_csv(TAB / "S_Table_hyperparameters_selected.csv", index=False)

MET = ["Accuracy", "Precision", "Sensitivity", "Specificity", "F1-score", "Balanced accuracy", "MCC", "ROC-AUC"]
summ = cv.groupby("model")[MET + ["train_ROC-AUC", "fit_seconds", "n_features"]].mean()
sd = cv.groupby("model")[MET].std()
summ["Generalization index"] = 1 - (summ["train_ROC-AUC"] - summ["ROC-AUC"]).clip(lower=0) / summ["train_ROC-AUC"]
summ["Stability index"] = 1 - sd["ROC-AUC"] / sd["ROC-AUC"].drop("Dummy (prior)").max()
summ = summ.loc[ALGOS]
tab2 = summ[MET].round(3).astype(str) + " ± " + sd.loc[ALGOS, MET].round(3).astype(str)
tab2.to_csv(TAB / "Table2_supervised_nestedCV.csv")
summ.to_csv(TAB / "supervised_nestedCV_summary.csv")
NUM["Table2"] = summ[MET + ["Generalization index", "Stability index", "n_features"]].round(3).to_dict(orient="index")
print(tab2.to_string())

In [ ]:
# independent test set: grouped 70/30 split, tuned on the 70% with grouped CV
gss = GroupShuffleSplit(n_splits=20, test_size=0.3, random_state=42)
for tr, te in gss.split(Xsup, ysup, gsup):                     # first split with both classes in proportion
    if abs(ysup[te].mean() - ysup.mean()) < 0.01:
        break
test_rows = []
for name in ALGOS:
    model, _ = fit_model(name, Xsup[tr], ysup[tr], gsup[tr], 42)
    r = {"model": name}; r.update(scores(ysup[te], model.predict_proba(Xsup[te])[:, 1])); test_rows.append(r)
tab3 = pd.DataFrame(test_rows).set_index("model").round(3)
tab3.to_csv(TAB / "Table3_independent_test.csv")
NUM["test_n"] = int(len(te)); NUM["test_tumour"] = int(ysup[te].sum()); NUM["Table3"] = tab3.to_dict(orient="index")
print(tab3[["Accuracy", "Balanced accuracy", "ROC-AUC"]].to_string())

In [ ]:
# Fig 4: UMAP before / after feature selection (visualization only; selection on all samples)
Xs_all = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(Xsup))
sel = ba.MWUSelector().fit(SimpleImputer(strategy="median").fit_transform(Xsup), ysup).support_
NUM["fig4_selected_features"] = int(sel.sum())
from sklearn.decomposition import PCA
emb_b = ba.umap_embed(PCA(50, random_state=0).fit_transform(Xs_all), 15, 0.1, 2, random_state=42)
emb_a = ba.umap_embed(PCA(min(50, sel.sum()), random_state=0).fit_transform(Xs_all[:, sel]), 15, 0.1, 2, random_state=42)
from sklearn.metrics import silhouette_score
NUM["fig4_silhouette_before"] = float(silhouette_score(emb_b, ysup)); NUM["fig4_silhouette_after"] = float(silhouette_score(emb_a, ysup))
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.3))
for ax, E, lbl, t in zip(axes, [emb_b, emb_a], "ab", ["All features", f"Selected features (n = {sel.sum()})"]):
    for cls, name in [(1, "Tumor"), (0, "Normal")]:
        m = ysup == cls
        ax.scatter(E[m, 0], E[m, 1], s=6 if cls else 12, c=COL[name], alpha=0.7 if cls else 0.9, edgecolors="white", linewidths=0.2,
                   label=f"{name} (n={m.sum()})", zorder=2 if cls else 3)
    ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2"); ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.02, 0.02, f"silhouette = {silhouette_score(E, ysup):.2f}", transform=ax.transAxes, fontsize=7, color=COL["ink2"])
    ba.panel_label(ax, lbl, x=-0.06)
axes[0].legend(loc="upper right", markerscale=2)
fig.tight_layout(); savefig(fig, "Fig4_umap_feature_selection", "Fig4")

In [ ]:
# Fig 5: confusion matrices (out-of-fold, seed 42): best classifier and decision tree
best_algo = summ.drop("Dummy (prior)")["ROC-AUC"].idxmax(); NUM["best_algo"] = best_algo
fig, axes = plt.subplots(1, 2, figsize=(6.2, 2.9))
for ax, name, lbl in zip(axes, [best_algo, "Decision Tree"], "ab"):
    o = oof[(oof.model == name) & (oof.seed == SEEDS[0])]
    cm = confusion_matrix(o.y, (o.prob >= 0.5).astype(int))
    pct = cm / cm.sum(1, keepdims=True) * 100          # row percentages (per true class)
    ax.imshow(cm / cm.sum(1, keepdims=True), cmap=ba.SEQ, vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            v = cm[i, j] / cm[i].sum()
            ax.text(j, i, f"{cm[i, j]}\n({pct[i, j]:.1f}%)", ha="center", va="center", fontsize=8,
                    color="white" if v > 0.6 else COL["ink"])
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Normal", "Tumor"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["Normal", "Tumor"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(name)
    for s in ax.spines.values(): s.set_visible(False)
    ba.panel_label(ax, lbl, x=-0.3)
fig.tight_layout(); savefig(fig, "Fig5_confusion_matrices", "Fig5")

# Fig 6: out-of-fold ROC curves (seed 42)
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for name in ALGOS:
    o = oof[(oof.model == name) & (oof.seed == SEEDS[0])]
    fpr, tpr, _ = roc_curve(o.y, o.prob)
    ax.plot(fpr, tpr, color=ALGO_COLORS[name], lw=1.2 if name != "Dummy (prior)" else 1.0, ls="-" if name != "Dummy (prior)" else "--",
            label=f"{name} ({summ.loc[name, 'ROC-AUC']:.3f})")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.set_xlim(-0.01, 1); ax.set_ylim(0, 1.01)
ax.legend(title="Mean ROC-AUC (nested CV)", loc="lower right", fontsize=6.5, title_fontsize=7)
fig.tight_layout(); savefig(fig, "Fig6_roc_curves", "Fig6")

# Fig 7: decision matrix
dm_rows = ["Accuracy", "Balanced accuracy", "Precision", "Sensitivity", "Specificity", "F1-score", "MCC", "ROC-AUC",
           "(Normalized fit time)$^{-1}$", "Generalization index", "Stability index"]
D = summ.drop("Dummy (prior)").copy()
D["(Normalized fit time)$^{-1}$"] = 1 - (D.fit_seconds - D.fit_seconds.min()) / (D.fit_seconds.max() - D.fit_seconds.min())
D["MCC"] = D["MCC"].clip(lower=0)
M = D[dm_rows].T
tot = M.sum(0)
order = tot.sort_values(ascending=False).index
M = M[order]; tot = tot[order]
NUM["decision_total"] = tot.round(3).to_dict(); NUM["decision_best"] = order[0]
fig, ax = plt.subplots(figsize=(6.8, 4.3))
Mshow = np.vstack([M.values, (tot / tot.max()).values[None, :]])
im = ax.imshow(Mshow, cmap=ba.SEQ, vmin=0, vmax=1, aspect="auto")
for i in range(Mshow.shape[0]):
    for j in range(Mshow.shape[1]):
        val = M.values[i, j] if i < len(dm_rows) else tot.values[j]
        ax.text(j, i, f"{val:.3f}" if i < len(dm_rows) else f"{val:.2f}", ha="center", va="center", fontsize=6.5,
                color="white" if Mshow[i, j] > 0.6 else COL["ink"], fontweight="bold" if i == len(dm_rows) else None)
ax.axhline(len(dm_rows) - 0.5, color=COL["ink"], lw=1.2)
ax.set_yticks(range(len(dm_rows) + 1)); ax.set_yticklabels(dm_rows + ["Total"])
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=35, ha="right")
for s in ax.spines.values(): s.set_visible(False)
cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02); cb.set_label("Score (1 = optimal)"); cb.outline.set_visible(False)
fig.tight_layout(); savefig(fig, "Fig7_decision_matrix", "Fig7")

## 4. Flux solutions compared (reviewer 1, comment 1)

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.feature_selection import SelectKBest, f_classif
sol_rows = []
for s in ["FBA", "pFBA", "L1w", "L2", "L2w"]:
    Xf = clean(lab[flux_cols([s])])
    tum = df[(df.sample_type == "Tumor") & df.is_first_tumor_vial]
    Xt = clean(tum[flux_cols([s])])
    cv_s = (Xt.std() / Xt.abs().mean()).replace([np.inf], np.nan)
    pipe = Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler()), ("k", SelectKBest(f_classif, k=min(100, Xf.shape[1]))),
                     ("c", LogisticRegression(C=0.1, class_weight="balanced", max_iter=4000))])
    aucs = []
    for seed in SEEDS:
        pr = cross_val_predict(pipe, Xf.values, ysup, groups=gsup, cv=StratifiedGroupKFold(5, shuffle=True, random_state=seed), method="predict_proba")[:, 1]
        aucs.append(roc_auc_score(ysup, pr))
    sol_rows.append({"solution": s, "reactions_with_flux": Xt.shape[1], "median_CV_across_tumours": float(cv_s.median()),
                     "tumour_normal_AUC": float(np.mean(aucs)), "AUC_sd": float(np.std(aucs))})
sol_tab = pd.DataFrame(sol_rows)
# similarity between solutions: correlation of per-tumor flux vectors (same reactions)
pairs = []
tum = df[(df.sample_type == "Tumor") & df.is_first_tumor_vial]
for a in ["FBA", "pFBA", "L1w", "L2", "L2w"]:
    for b2 in ["FBA", "pFBA", "L1w", "L2", "L2w"]:
        A = tum[flux_cols([a])].fillna(0).values; Bm = tum[flux_cols([b2])].fillna(0).values
        r = [np.corrcoef(A[i], Bm[i])[0, 1] for i in range(0, len(A), 5)]
        pairs.append({"a": a, "b": b2, "r": float(np.nanmedian(r))})
sol_corr = pd.DataFrame(pairs).pivot(index="a", columns="b", values="r").loc[["FBA", "pFBA", "L1w", "L2", "L2w"], ["FBA", "pFBA", "L1w", "L2", "L2w"]]
sol_tab.to_csv(TAB / "S_Table_flux_solutions.csv", index=False); sol_corr.to_csv(TAB / "flux_solution_correlation.csv")
NUM["solutions"] = sol_tab.round(3).to_dict(orient="records"); NUM["solution_corr"] = sol_corr.round(3).to_dict()
fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.6))
ax = axes[0]; ax.bar(sol_tab.solution, sol_tab.reactions_with_flux, color=[COL["LumA"] if s in SOLS else COL["muted"] for s in sol_tab.solution])
ax.set_ylabel("Reactions carrying flux\n(≥5% of tumors)"); ba.panel_label(ax, "a", x=-0.3)
ax = axes[1]; ax.bar(sol_tab.solution, sol_tab.tumour_normal_AUC, yerr=sol_tab.AUC_sd, capsize=2,
                     color=[COL["LumA"] if s in SOLS else COL["muted"] for s in sol_tab.solution])
ax.set_ylim(0.5, 1); ax.set_ylabel("Tumor vs normal ROC-AUC"); ba.panel_label(ax, "b", x=-0.3)
ax = axes[2]; im = ax.imshow(sol_corr.values, cmap=ba.SEQ, vmin=0, vmax=1)
ax.set_xticks(range(5)); ax.set_yticks(range(5)); ax.set_xticklabels(sol_corr.columns, rotation=45); ax.set_yticklabels(sol_corr.index)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{sol_corr.values[i, j]:.2f}", ha="center", va="center", fontsize=6, color="white" if sol_corr.values[i, j] > 0.6 else COL["ink"])
for s in ax.spines.values(): s.set_visible(False)
ax.set_title("Median per-model correlation", fontsize=8); ba.panel_label(ax, "c", x=-0.35)
fig.tight_layout(); savefig(fig, "S1_Fig_flux_solutions", "S1_Fig")

## 5. Unsupervised clustering and clinical concordance (Table 4, Figs 8-10)

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, HDBSCAN, MeanShift, estimate_bandwidth
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score, adjusted_mutual_info_score
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

tum = df[(df.sample_type == "Tumor") & df.is_first_tumor_vial].reset_index(drop=True)
NUM["unsup_tumours"] = int(len(tum))
FEATSETS = {"FBA": ["FBA"], "pFBA": ["pFBA"], "L1w": ["L1w"], "FBA+pFBA+L1w": SOLS}
Xmet = {}
for k, s in FEATSETS.items():
    X = clean(tum[metric_cols(s)])
    X = np.sign(X) * np.log1p(X.abs())                     # signed log: metrics span orders of magnitude
    Xmet[k] = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(X))

def embeddings(X, tag):
    E = {}
    Xp = PCA(min(50, X.shape[1]), random_state=0).fit_transform(X)
    for nc in (2, 3):
        E[f"{tag}|PCA|C{nc}"] = Xp[:, :nc]
    for nn in ([15, 30] if not FAST else [15]):
        for nc in (2, 3):
            for seed in SEEDS:
                E[f"{tag}|UMAP|N{nn}_C{nc}_S{seed}"] = ba.umap_embed(Xp, nn, 0.1, nc, random_state=seed)
    return E

def algorithms(Xe):
    nn = np.sort(((Xe[:, None, :] - Xe[None, :200, :]) ** 2).sum(-1) ** 0.5, axis=1)[:, 5]
    eps = np.quantile(nn, [0.5, 0.75, 0.9])
    bw = estimate_bandwidth(Xe, quantile=0.2, n_samples=min(500, len(Xe)), random_state=0) or 1.0
    return {"KMeans": (KMeans, ParameterGrid({"n_clusters": range(2, 7), "n_init": [10]}), True),
            "Agglomerative": (AgglomerativeClustering, ParameterGrid({"n_clusters": range(2, 7), "linkage": ["ward", "average"]}), False),
            "GMM": (GaussianMixture, ParameterGrid({"n_components": range(2, 6)}), True),
            "BayesGMM": (BayesianGaussianMixture, ParameterGrid({"n_components": range(2, 6)}), True),
            "DBSCAN": (DBSCAN, ParameterGrid({"eps": list(eps), "min_samples": [10]}), False),
            "HDBSCAN": (HDBSCAN, ParameterGrid({"min_cluster_size": [20, 40]}), False),
            "MeanShift": (MeanShift, ParameterGrid({"bandwidth": [bw * 0.5, bw, bw * 1.5]}), False)}

def sweep(E, Xfeat_of, min_frac=0.02, max_noise=5.0):
    out = []
    for ek, Xe in E.items():
        for an, (cls, grid, stoch) in algorithms(Xe).items():
            best = None
            for seed in (SEEDS if stoch else SEEDS[:1]):
                for p in grid:
                    try:
                        lab_ = np.asarray((cls(**p, random_state=seed) if stoch else cls(**p)).fit_predict(Xe))
                    except Exception:
                        continue
                    ok = lab_ != -1; sz = pd.Series(lab_[ok]).value_counts()
                    if 100 * (~ok).mean() > max_noise or len(sz) < 2 or sz.min() < max(10, min_frac * len(lab_)):
                        continue
                    s = silhouette_score(Xe[ok], lab_[ok])
                    if best is None or s > best["silhouette_embedding"]:
                        best = {"embedding": ek, "reduction": ek.split("|")[1], "features": ek.split("|")[0], "algorithm": an,
                                "params": str(p), "seed": seed, "k": len(sz), "smallest": int(sz.min()), "noise_pct": 100 * (~ok).mean(),
                                "silhouette_embedding": s, "davies_bouldin": davies_bouldin_score(Xe[ok], lab_[ok]),
                                "calinski_harabasz": calinski_harabasz_score(Xe[ok], lab_[ok]), "labels": lab_}
            if best:
                Xf = Xfeat_of(best["features"]); ok = best["labels"] != -1
                best["silhouette_features"] = silhouette_score(Xf[ok], best["labels"][ok])
                out.append(best)
    return pd.DataFrame(out).sort_values("silhouette_embedding", ascending=False).reset_index(drop=True)

t0 = time.time()
E_met = {}
for k in FEATSETS:
    E_met.update(embeddings(Xmet[k], k))
res_met = sweep(E_met, lambda f: Xmet[f])
print(f"metabolic sweep: {len(res_met)} solutions ({(time.time() - t0) / 60:.1f} min)")

# artefact checks for every solution
def artefacts(labels, frame):
    ok = labels != -1; out = {}
    for k, v in {"missing_metadata": ~frame.in_metadata.fillna(False).astype(bool), "tissue_source_site": frame.tss_code}.items():
        out[f"V_{k}"], out[f"p_{k}"] = cramers_v(labels[ok], np.asarray(v)[ok])
    return out
if "MediumSupplement" in qc:
    tum["medium_supplemented"] = tum["sample"].map(qc.set_index("sample").MediumSupplement.fillna("").ne(""))
else:
    tum["medium_supplemented"] = False
def artefacts(labels, frame):
    ok = labels != -1; out = {}
    covs = {"missing_metadata": ~frame.in_metadata.fillna(False).astype(bool), "tissue_source_site": frame.tss_code,
            "medium_supplemented": frame.medium_supplemented}
    for k, v in covs.items():
        out[f"V_{k}"], out[f"p_{k}"] = cramers_v(labels[ok], np.asarray(v)[ok])
    out["p_model_size"] = kruskal(*[frame.nRxns.values[ok][labels[ok] == c] for c in np.unique(labels[ok])]).pvalue
    return out
from scipy.stats import kruskal
art = pd.DataFrame([artefacts(l, tum) for l in res_met["labels"]])
res_met = pd.concat([res_met, art], axis=1)
_f_meta = (res_met.V_missing_metadata >= 0.3) | (res_met.p_missing_metadata < 1e-3)
_f_tss = (res_met.V_tissue_source_site >= 0.3) | (res_met.p_tissue_source_site < 1e-3)
_f_sup = (res_met.V_medium_supplemented.fillna(0) >= 0.3)
res_met["artefact_flag"] = _f_meta | _f_tss | _f_sup
NUM["sweep_counts"] = {"evaluated": int(len(res_met)), "flag_missing_metadata": int(_f_meta.sum()),
                       "flag_tissue_source_site": int(_f_tss.sum()), "flag_medium_supplemented": int(_f_sup.sum()),
                       "artifact_free": int((~res_met.artefact_flag).sum())}

def stability(res):
    """mean ARI of the same algorithm/feature set/UMAP setting across embedding seeds (PCA is deterministic -> 1)."""
    key = res.embedding.str.replace(r"_S\d+$", "", regex=True) + "|" + res.algorithm
    out = np.ones(len(res))
    for k_, idx in res.groupby(key).groups.items():
        idx = list(idx)
        if len(idx) < 2:
            out[[res.index.get_loc(i) for i in idx]] = np.nan if "UMAP" in k_ else 1.0
            continue
        a_ = [adjusted_rand_score(res.loc[i, "labels"], res.loc[j, "labels"]) for ii_, i in enumerate(idx) for j in idx[ii_ + 1:]]
        out[[res.index.get_loc(i) for i in idx]] = float(np.mean(a_))
    return out
res_met["stability_ARI"] = stability(res_met)
print(NUM["sweep_counts"])
res_met.drop(columns="labels").to_csv(TAB / "clustering_metabolic_sweep.csv", index=False)

In [ ]:
# clinical clustering (tumors with MetaData; missing categories would otherwise drive the clusters)
CLIN_CAT = ["stage", "T", "N", "M", "histology", "ER", "PR", "HER2", "receptor_subtype", "menopausal_status", "genetic_ancestry"]
CLIN_NUM = ["age_years"]
tc_mask = tum.in_metadata.fillna(False).astype(bool).values
tc = tum[tc_mask].reset_index(drop=True)
ct = ColumnTransformer([("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), CLIN_NUM),
                        ("cat", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="missing")),
                                          ("o", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CLIN_CAT)])
Xclin = ct.fit_transform(tc[CLIN_NUM + CLIN_CAT].astype({c: object for c in CLIN_CAT}).where(tc[CLIN_NUM + CLIN_CAT].notna(), np.nan))
E_clin = embeddings(Xclin, "clinical")
res_clin = sweep(E_clin, lambda f: Xclin)
res_clin["stability_ARI"] = stability(res_clin)
res_clin.drop(columns="labels").to_csv(TAB / "clustering_clinical_sweep.csv", index=False)
NUM["clin_tumours"] = int(len(tc))

# Table 4: best silhouette by reduction
t4 = pd.DataFrame({"Clinical data": res_clin.groupby("reduction").silhouette_embedding.max(),
                   "Metabolic data": res_met[~res_met.artefact_flag].groupby("reduction").silhouette_embedding.max()}).T
t4_feat = pd.DataFrame({"Clinical data": res_clin.groupby("reduction").silhouette_features.max(),
                        "Metabolic data": res_met[~res_met.artefact_flag].groupby("reduction").silhouette_features.max()}).T
t4.round(2).to_csv(TAB / "Table4_silhouette.csv"); t4_feat.round(2).to_csv(TAB / "Table4_silhouette_feature_space.csv")
NUM["Table4"] = t4.round(3).to_dict(); NUM["Table4_feature_space"] = t4_feat.round(3).to_dict()
print(t4.round(3)); print(t4_feat.round(3))

In [ ]:
# selected solutions and concordance
# pre-defined protocol: artifact-free, stable across seeds (ARI >= 0.5), FBA-only feature set excluded because FBA
# optima are degenerate; rank by silhouette in the original feature space, then in the embedding.
def select(res, exclude_fba=True):
    c_ = res[(~res.get("artefact_flag", pd.Series(False, index=res.index))) & (res.stability_ARI.fillna(0) >= 0.5)]
    if exclude_fba:
        c_ = c_[c_.features != "FBA"]
    return c_.sort_values(["silhouette_features", "silhouette_embedding"], ascending=False).iloc[0]
sel_met = select(res_met); sel_clin = select(res_clin, exclude_fba=False)
_fba = res_met[(~res_met.artefact_flag) & (res_met.features == "FBA")].iloc[0]          # sensitivity: FBA-only
_fs = pd.Series(_fba["labels"]).value_counts()
NUM["fba_partition"] = {"algorithm": _fba["algorithm"], "k": int(_fba["k"]), "smallest": int(_fs.min()),
                        "silhouette_embedding": float(_fba["silhouette_embedding"]), "silhouette_features": float(_fba["silhouette_features"]),
                        "ARI_with_selected": float(adjusted_rand_score(_fba["labels"], sel_met["labels"]))}
tum["metabolic_cluster"] = sel_met["labels"]
tc["clinical_cluster"] = sel_clin["labels"]
NUM["sel_met"] = {k: (str(v) if not isinstance(v, (int, float, np.integer, np.floating)) else float(v)) for k, v in sel_met.drop("labels").items()}
NUM["sel_clin"] = {k: (str(v) if not isinstance(v, (int, float, np.integer, np.floating)) else float(v)) for k, v in sel_clin.drop("labels").items()}
NUM["met_cluster_sizes"] = tum.metabolic_cluster.value_counts().sort_index().to_dict()
NUM["clin_cluster_sizes"] = tc.clinical_cluster.value_counts().sort_index().to_dict()
pos = tum.reset_index().set_index("sample").loc[tc["sample"], "index"].values
pairs = []
for i, rc in res_clin.iterrows():
    for j, rm in res_met[~res_met.artefact_flag].iterrows():
        a, b2 = rc["labels"], rm["labels"][pos]; ok = (a != -1) & (b2 != -1)
        pairs.append({"clinical": i, "metabolic": j, "ARI": adjusted_rand_score(a[ok], b2[ok]), "AMI": adjusted_mutual_info_score(a[ok], b2[ok])})
ari = pd.DataFrame(pairs); ari.to_csv(TAB / "concordance_ARI_AMI.csv", index=False)
NUM["ARI_max"] = float(ari.ARI.max()); NUM["ARI_median"] = float(ari.ARI.median()); NUM["AMI_max"] = float(ari.AMI.max())
NUM["ARI_selected"] = float(adjusted_rand_score(sel_clin["labels"], sel_met["labels"][pos]))
NUM["ARI_met_vs_PAM50"] = float(adjusted_rand_score(tum.PAM50[tum.PAM50.notna()], tum.metabolic_cluster[tum.PAM50.notna()]))
NUM["ARI_clin_vs_PAM50"] = float(adjusted_rand_score(tc.PAM50[tc.PAM50.notna()], tc.clinical_cluster[tc.PAM50.notna()]))
NUM["ARI_common_n"] = int(len(tc))
_rng = np.random.default_rng(0); _a = sel_clin["labels"]; _b = sel_met["labels"][pos]
_null = np.array([adjusted_rand_score(_a, _rng.permutation(_b)) for _ in range(1000)])
NUM["ARI_selected_perm_p"] = float((1 + (_null >= NUM["ARI_selected"]).sum()) / 1001)
NUM["ARI_null_95"] = float(np.quantile(_null, 0.95))
# non-circular clinical clustering: receptor variables removed
_cat2 = [c for c in CLIN_CAT if c not in ("ER", "PR", "HER2", "receptor_subtype")]
_ct2 = ColumnTransformer([("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), CLIN_NUM),
                          ("cat", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="missing")),
                                            ("o", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), _cat2)])
_X2 = _ct2.fit_transform(tc[CLIN_NUM + _cat2].astype({c: object for c in _cat2}).where(tc[CLIN_NUM + _cat2].notna(), np.nan))
_l2 = KMeans(n_clusters=int(sel_clin["k"]), n_init=10, random_state=42).fit_predict(_X2)
NUM["ARI_clin_noreceptor_vs_PAM50"] = float(adjusted_rand_score(tc.PAM50[tc.PAM50.notna()], _l2[tc.PAM50.notna().values]))
NUM["multi_tumor_patients"] = int(df[df.sample_type == "Tumor"].patient.duplicated().sum())
print({k: NUM[k] for k in ["ARI_max", "ARI_median", "ARI_selected", "ARI_met_vs_PAM50", "ARI_clin_vs_PAM50"]})

def short(r):
    p = re.sub(r"[{}' ]", "", r["params"]).replace("n_clusters:", "k=").replace("n_components:", "k=").replace(",n_init:10", "")
    return f"{r['algorithm']} · {r['reduction']}{'' if r['features'] == 'clinical' else ' · ' + r['features']} (k={r['k']})"

def distinct(res, n=10, thr=0.9):
    keep = []
    for i, r in res.iterrows():
        if all(adjusted_rand_score(r["labels"], res.loc[j, "labels"]) < thr for j in keep):
            keep.append(i)
        if len(keep) == n:
            break
    return keep
topc = distinct(res_clin); topm = distinct(res_met[~res_met.artefact_flag])
A = ari.pivot(index="clinical", columns="metabolic", values="ARI").loc[topc, topm]
fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(topc) + 2.6))
_v = max(0.1, np.abs(A.values).max())
im = ax.imshow(A.values, cmap=ba.DIV, vmin=-_v, vmax=_v)
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        ax.text(j, i, f"{A.values[i, j]:.3f}", ha="center", va="center", fontsize=6, color="white" if abs(A.values[i, j]) > 0.6 * _v else COL["ink"])
ax.set_yticks(range(len(topc))); ax.set_yticklabels([short(res_clin.loc[i]) for i in topc], fontsize=6.5)
ax.set_xticks(range(len(topm))); ax.set_xticklabels([short(res_met.loc[j]) for j in topm], rotation=40, ha="right", fontsize=6.5)
ax.set_ylabel("Clinical clustering"); ax.set_xlabel("Metabolic clustering")
for s in ax.spines.values(): s.set_visible(False)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02); cb.set_label("Adjusted Rand Index"); cb.outline.set_visible(False)
fig.tight_layout(); savefig(fig, "Fig8_concordance_ARI", "Fig8")

In [ ]:
# Figs 9-10: 2-D UMAP views colored by cluster (a) and by PAM50 subtype (b)
def view2d(E_dict, key, X):
    k2 = key.replace("_C3_", "_C2_").replace("|C3", "|C2")
    return E_dict[k2] if k2 in E_dict else ba.umap_embed(PCA(min(50, X.shape[1]), random_state=0).fit_transform(X), 15, 0.1, 2, random_state=42)

PAM_ORDER = ["LumA", "LumB", "Her2", "Basal", "Normal"]
def pam_color(v):
    return COL["Normal-like"] if v == "Normal" else COL.get(v, COL["muted"])

def cluster_fig(Y, labels, frame, title, fname, pname):
    fig, axes = plt.subplots(1, 2, figsize=(7.5, 4.0))
    ax = axes[0]
    for i, c in enumerate(sorted(set(labels))):
        m = labels == c
        ax.scatter(Y[m, 0], Y[m, 1], s=6, c=CLUSTER_COLORS[i % 6] if c != -1 else COL["muted"], alpha=0.8, edgecolors="none",
                   label=("noise" if c == -1 else f"Cluster {c + 1}") + f" (n={m.sum()})")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, markerscale=2.5, fontsize=6.5, columnspacing=1.0,
              handletextpad=0.3); ax.set_title(title, fontsize=8)
    ax = axes[1]
    pam = frame.PAM50.fillna("NA").values
    for v in ["NA"] + PAM_ORDER:
        m = pam == v
        if m.any():
            ax.scatter(Y[m, 0], Y[m, 1], s=6, c=pam_color(v) if v != "NA" else "#d9d8d3", alpha=0.85, edgecolors="none",
                       label=("Normal-like" if v == "Normal" else ("not available" if v == "NA" else v)) + f" (n={m.sum()})")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, markerscale=2.5, fontsize=6.5, columnspacing=1.0,
              handletextpad=0.3); ax.set_title("PAM50 subtype", fontsize=8)
    for a, l in zip(axes, "ab"):
        a.set_xticks([]); a.set_yticks([]); a.set_ylabel("UMAP2"); ba.panel_label(a, l, x=-0.06)
        a.text(1.0, -0.02, "UMAP1", transform=a.transAxes, ha="right", va="top", fontsize=7, color=COL["ink2"])
    fig.tight_layout(); savefig(fig, fname, pname)

Yc = view2d(E_clin, sel_clin["embedding"], Xclin)
cluster_fig(Yc, sel_clin["labels"], tc, f"Clinical clustering: {short(sel_clin)}", "Fig9_clinical_clusters", "Fig9")
Ym = view2d(E_met, sel_met["embedding"], Xmet[sel_met["features"]])
cluster_fig(Ym, sel_met["labels"], tum, f"Metabolic clustering: {short(sel_met)}", "Fig10_metabolic_clusters", "Fig10")

## 6. Characterization of the metabolic clusters (Fig 11)

In [ ]:
Xraw = clean(tum[metric_cols(SOLS)])
labm = tum.metabolic_cluster.values
eff = []
for c in sorted(set(labm) - {-1}):
    A_, B_ = Xraw[labm == c].fillna(0).values, Xraw[(labm != c) & (labm != -1)].fillna(0).values
    d, p = cliffs_matrix(A_, B_)
    eff.append(pd.DataFrame({"cluster": c + 1, "feature": Xraw.columns, "cliffs_delta": d, "p": p,
                             "median_cluster": np.median(A_, 0), "median_rest": np.median(B_, 0)}))
eff = pd.concat(eff); eff["FDR"] = eff.groupby("cluster").p.transform(bh)
eff["magnitude"] = pd.cut(eff.cliffs_delta.abs(), [-1, 0.147, 0.33, 0.474, 1], labels=["negligible", "small", "medium", "large"])
eff.to_csv(TAB / "S_Table_metabolic_cluster_effect_sizes.csv", index=False)
NUM["cluster_sig_features"] = eff[eff.FDR < 0.05].groupby("cluster").size().to_dict()
NUM["cluster_large_effects"] = eff[(eff.FDR < 0.05) & (eff.magnitude == "large")].groupby("cluster").size().to_dict()
sizes_ = pd.Series(labm[labm != -1]).value_counts()
tgt = int(sizes_.idxmin()) + 1                          # smallest cluster vs all other tumors
NUM["fig11_cluster"] = int(tgt)
e = eff[(eff.cluster == tgt) & (eff.FDR < 0.05)].assign(a=lambda x: x.cliffs_delta.abs()).sort_values("a", ascending=False).head(40).sort_values("cliffs_delta")
METRIC_NAMES = {"CU": "Carbon uptake", "EA": "Enzymatic abundance", "GlnDependence": "Glutamine dependence",
                "RedoxIndex": "Redox index", "ATPConsumption": "ATP consumption", "ATPProduction": "ATP production",
                "NADPHdemand": "NADPH demand", "AnabolismScore": "Anabolism score", "TCA completeness": "TCA cycle completeness",
                "WarburgIndex": "Warburg index", "LipidSat": "Saturated lipid synthesis", "LipidUnsat": "Unsaturated lipid synthesis",
                "LipidPL": "Phospholipid synthesis", "Exchange demand reaction": "Exchange/demand reactions",
                "Glycolysis gluconeogenesis": "Glycolysis/gluconeogenesis"}
def pretty(f):
    m = re.search(r"_(FBA|pFBA|L1w)$", f); sol = f" ({m.group(1)})" if m else ""
    f = re.sub(r"_(FBA|pFBA|L1w)$", "", f); f = re.sub(r"^SA_", "", f); f = f.replace("Oncomet_", "Secretion: ").replace("_", " ")
    return METRIC_NAMES.get(f, f) + sol
fig, ax = plt.subplots(figsize=(5.6, max(3.0, 0.15 * len(e) + 0.8)))
cmap = plt.get_cmap(ba.DIV)
ax.hlines(np.arange(len(e)), 0, e.cliffs_delta, color=[cmap(0.15) if v < 0 else cmap(0.85) for v in e.cliffs_delta], lw=1.2)
ax.scatter(e.cliffs_delta, np.arange(len(e)), s=16, c=[cmap(0.15) if v < 0 else cmap(0.85) for v in e.cliffs_delta], zorder=3)
for t_ in (0.147, 0.33, 0.474):
    for sgn in (-1, 1):
        ax.axvline(sgn * t_, color=COL["grid"], lw=0.6, zorder=0)
ax.axvline(0, color=COL["ink2"], lw=0.6)
ax.set_yticks(np.arange(len(e))); ax.set_yticklabels([pretty(f) for f in e.feature], fontsize=6)
ax.set_xlim(-1, 1); ax.set_xlabel(f"Cliff's δ (cluster {tgt} vs other tumors)")
ax.text(-0.98, len(e) - 0.2, "lower in cluster", fontsize=6.5, color=cmap(0.1), va="bottom")
ax.text(0.98, len(e) - 0.2, "higher in cluster", fontsize=6.5, color=cmap(0.9), va="bottom", ha="right")
fig.tight_layout(); savefig(fig, "Fig11_cliffs_delta", "Fig11")

# cluster vs clinical variables
assoc = []
okc = tum.metabolic_cluster != -1
for v in CLIN_CAT + ["tss_code", "race", "sample_type"]:
    s_ = tum[okc & tum[v].notna()]
    if s_[v].nunique() > 1:
        V, p = cramers_v(s_.metabolic_cluster, s_[v]); assoc.append({"variable": v, "n": len(s_), "cramers_v": V, "p": p})
from scipy.stats import kruskal
s_ = tum[okc & tum.age_years.notna()]
assoc.append({"variable": "age_years", "n": len(s_), "cramers_v": np.nan, "p": kruskal(*[g.age_years for _, g in s_.groupby("metabolic_cluster")]).pvalue})
assoc = pd.DataFrame(assoc); assoc["FDR"] = bh(assoc.p.values)
assoc.sort_values("p").to_csv(TAB / "S_Table_metabolic_cluster_clinical_associations.csv", index=False)
NUM["cluster_assoc"] = assoc.set_index("variable")[["cramers_v", "p", "FDR"]].round(4).to_dict(orient="index")
ct_pam = pd.crosstab(tum.metabolic_cluster + 1, tum.PAM50.fillna("NA")); ct_pam.to_csv(TAB / "metabolic_cluster_vs_PAM50.csv")
ct_er = pd.crosstab(tum.metabolic_cluster + 1, tum.ER.fillna("NA")); ct_er.to_csv(TAB / "metabolic_cluster_vs_ER.csv")
NUM["cluster_x_PAM50"] = ct_pam.to_dict(); NUM["cluster_x_ER"] = ct_er.to_dict()
print(assoc.sort_values("p").head(8).to_string()); print(ct_pam)

## 7. Clinical variables encoded in metabolic fluxes (Fig 12)

In [ ]:
TASKS = {
    "ER status (+ vs −)": tum.ER.map({"positive": 1, "negative": 0}),
    "PR status (+ vs −)": tum.PR.map({"positive": 1, "negative": 0}),
    "HER2 status (+ vs −)": tum.HER2.map({"positive": 1, "negative": 0}),
    "Triple negative vs other": tum.receptor_subtype.map(lambda v: np.nan if pd.isna(v) else int(v == "TNBC")),
    "PAM50 Basal vs other": tum.PAM50.map(lambda v: np.nan if pd.isna(v) else int(v == "Basal")),
    "PAM50 LumA vs LumB": tum.PAM50.map({"LumA": 1, "LumB": 0}),
    "PAM50 Her2 vs other": tum.PAM50.map(lambda v: np.nan if pd.isna(v) else int(v == "Her2")),
    "Lobular vs ductal": tum.histology.map({"ILC": 1, "IDC": 0}),
    "Stage III–IV vs I–II": tum.stage.map({"I": 0, "II": 0, "III": 1, "IV": 1}),
    "Node positive vs negative": tum.N.map(lambda v: np.nan if pd.isna(v) else int(v != "N0")),
    "Age ≥ 60 years": (tum.age_years >= 60).astype(float).where(tum.age_years.notna()),
    "Post- vs premenopausal": tum.menopausal_status.map({"postmenopausal": 1, "premenopausal": 0}),
    "African vs European ancestry": tum.genetic_ancestry.map(lambda v: 1 if str(v).startswith("African") else (0 if str(v).startswith("European") else np.nan)),
}
FS = {"Derived metrics": clean(tum[metric_cols(SOLS)]), "Reaction fluxes": clean(tum[flux_cols(SOLS)])}
enc = []
for name, yv in TASKS.items():
    m = yv.notna().values; y_ = yv[m].astype(int).values
    for fs, Xdf in FS.items():
        X_ = Xdf.loc[m].values
        pipe = Pipeline([("i", SimpleImputer(strategy="median")), ("v", ba.MWUSelector(alpha=0.05, k_min=20)),
                         ("s", StandardScaler()), ("c", LogisticRegression(C=0.05, class_weight="balanced", max_iter=4000))])
        aucs = []
        for seed in SEEDS:
            from sklearn.model_selection import StratifiedKFold
            pr = cross_val_predict(pipe, X_, y_, cv=StratifiedKFold(5, shuffle=True, random_state=seed), method="predict_proba")[:, 1]
            aucs.append(roc_auc_score(y_, pr))
        enc.append({"task": name, "features": fs, "n": int(len(y_)), "positives": int(y_.sum()), "AUC": float(np.mean(aucs)), "AUC_sd": float(np.std(aucs))})
enc = pd.DataFrame(enc); enc.to_csv(TAB / "clinical_variables_from_fluxes.csv", index=False)
NUM["encoding"] = enc.round(3).to_dict(orient="records")
print(enc.pivot(index="task", columns="features", values="AUC").round(3))
order_t = enc[enc.features == "Reaction fluxes"].sort_values("AUC").task.tolist()
fig, ax = plt.subplots(figsize=(5.6, 4.0))
yy = np.arange(len(order_t)); h = 0.38
for k_, (fs, colr) in enumerate([("Derived metrics", COL["muted"]), ("Reaction fluxes", COL["LumA"])]):
    d_ = enc[enc.features == fs].set_index("task").loc[order_t]
    ax.barh(yy + (k_ - 0.5) * h, d_.AUC - 0.5, left=0.5, height=h, color=colr, xerr=d_.AUC_sd, error_kw={"lw": 0.6, "capsize": 1.5, "ecolor": COL["ink2"]}, label=fs)
ax.axvline(0.5, color=COL["ink2"], lw=0.7, ls="--"); ax.text(0.505, len(order_t) - 0.4, "chance", fontsize=6.5, color=COL["ink2"])
ax.set_yticks(yy); ax.set_yticklabels([f"{t} (n={enc[enc.task == t].n.iloc[0]})" for t in order_t])
ax.set_xlim(0.4, 1.0); ax.set_xlabel("ROC-AUC (5-fold CV × 3 seeds)")
ax.legend(loc="lower right"); ax.grid(axis="x", color=COL["grid"], lw=0.6); ax.set_axisbelow(True)
fig.tight_layout(); savefig(fig, "Fig12_clinical_encoding", "Fig12")

## 8. Metabolic signatures across PAM50 subtypes (Fig 13)

In [ ]:
Xm = clean(tum[metric_cols(SOLS)]); pam = tum.PAM50
okp = pam.isin(PAM_ORDER).values
kw = []
for f in Xm.columns:
    groups = [Xm.loc[okp & (pam == g).values, f].dropna().values for g in PAM_ORDER]
    groups = [g for g in groups if len(g) > 3]
    try:
        H, p = kruskal(*groups)
    except ValueError:
        H, p = 0.0, 1.0
    n_ = sum(len(g) for g in groups)
    kw.append({"feature": f, "H": H, "p": p, "eta2": max(0.0, (H - len(groups) + 1) / (n_ - len(groups)))})
kw = pd.DataFrame(kw); kw["FDR"] = bh(kw.p.values)
kw.sort_values("p").to_csv(TAB / "S_Table_PAM50_metabolic_features.csv", index=False)
NUM["pam50_sig_features"] = int((kw.FDR < 0.05).sum()); NUM["pam50_tested"] = int(len(kw))
_k = kw.assign(base=kw.feature.str.replace(r"_(FBA|pFBA|L1w)$", "", regex=True), sol=kw.feature.str.extract(r"_(FBA|pFBA|L1w)$")[0])
_sig = _k[_k.FDR < 0.05].groupby("base").sol.nunique()
NUM["pam50_sig_all3"] = int((_sig == 3).sum()); NUM["pam50_sig_ge2"] = int((_sig >= 2).sum()); NUM["pam50_sig_bases"] = int(len(_sig))
topf = kw[kw.FDR < 0.05].sort_values("eta2", ascending=False).feature.tolist()
topf = [f for f in topf if Xm[f].nunique() > 20][:40]
Z = pd.DataFrame(StandardScaler().fit_transform(np.sign(Xm[topf]) * np.log1p(Xm[topf].abs())), columns=topf)
Z["PAM50"] = pam.values
H_ = Z[okp].groupby("PAM50")[topf].mean().loc[[p for p in PAM_ORDER if p in set(pam)]]
H_ = H_ - H_.mean(0)                                    # relative to the average subtype
order_f = H_.T.apply(lambda r: np.argmax(r.values) * 10 + r.max(), axis=1).sort_values().index   # group features by the subtype where they peak
H_ = H_[order_f]
fig = plt.figure(figsize=(6.0, 7.2))
gs = fig.add_gridspec(2, 2, height_ratios=[0.035, 1], width_ratios=[1, 0.04], hspace=0.02, wspace=0.05)
axc = fig.add_subplot(gs[0, 0]); ax = fig.add_subplot(gs[1, 0]); cax = fig.add_subplot(gs[1, 1])
axc.imshow([[i for i in range(len(H_.index))]], cmap=plt.matplotlib.colors.ListedColormap([pam_color(p) for p in H_.index]), aspect="auto")
axc.set_xticks([]); axc.set_yticks([])
for s in axc.spines.values(): s.set_visible(False)
vmax = float(np.nanquantile(np.abs(H_.values), 0.98))
im = ax.imshow(H_.T.values, cmap=ba.DIV, vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_xticks(range(len(H_.index))); ax.set_xticklabels([("Normal-like" if p == "Normal" else p) + f"\n(n={(pam == p).sum()})" for p in H_.index])
ax.set_yticks(range(len(order_f))); ax.set_yticklabels([pretty(f) for f in order_f], fontsize=6)
for s in ax.spines.values(): s.set_visible(False)
cb = fig.colorbar(im, cax=cax); cb.set_label("Mean z-score (relative to all subtypes)"); cb.outline.set_visible(False)
fig.subplots_adjust(left=0.42, right=0.9, top=0.97, bottom=0.08)
savefig(fig, "Fig13_PAM50_signatures", "Fig13")

## 9. Survival (supplementary) and Pareto surfaces

In [ ]:
s_ = tum[(tum.metabolic_cluster != -1) & tum.OS_days.notna() & tum.OS_event.notna()].copy()
s_["years"] = s_.OS_days / 365.25
stat, dfree, p_lr = ba.logrank(s_.years, s_.OS_event.astype(int), s_.metabolic_cluster)
NUM["logrank_clusters"] = {"chi2": stat, "df": dfree, "p": p_lr}
sp_ = tum[tum.PAM50.isin(PAM_ORDER) & tum.OS_days.notna()].copy(); sp_["years"] = sp_.OS_days / 365.25
stat2, d2, p_lr2 = ba.logrank(sp_.years, sp_.OS_event.astype(int), sp_.PAM50)
NUM["logrank_PAM50"] = {"chi2": stat2, "df": d2, "p": p_lr2}
cx = s_[["years", "OS_event", "age_years", "stage", "metabolic_cluster", "PAM50"]].dropna()
cx = cx[cx.PAM50.isin(PAM_ORDER)]
Xc = pd.get_dummies(cx[["stage", "metabolic_cluster", "PAM50"]].astype(str), drop_first=True, dtype=float)
NUM["surv_n"] = int(len(s_)); NUM["surv_events"] = int(s_.OS_event.sum()); NUM["cox_n"] = int(len(cx)); NUM["cox_events"] = int(cx.OS_event.sum())
Xc["age_per_10y"] = cx.age_years / 10
cox = ba.cox_ph(cx.years, cx.OS_event.astype(int), Xc.values, list(Xc.columns)); cox.to_csv(TAB / "S_Table_cox_metabolic_clusters.csv", index=False)
NUM["cox"] = cox.round(3).to_dict(orient="records")
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.2))
for ax, frame, gcol, colors, title, p_ in [(axes[0], s_, "metabolic_cluster", {c: CLUSTER_COLORS[i] for i, c in enumerate(sorted(s_.metabolic_cluster.unique()))}, "Metabolic clusters", p_lr),
                                            (axes[1], sp_, "PAM50", {p: pam_color(p) for p in PAM_ORDER}, "PAM50 subtypes (reference)", p_lr2)]:
    for gval, g in frame.groupby(gcol):
        tt, S_, lo, hi = ba.kaplan_meier(g.years, g.OS_event.astype(int))
        lbl = (f"Cluster {gval + 1}" if gcol == "metabolic_cluster" else ("Normal-like" if gval == "Normal" else gval)) + f" (n={len(g)})"
        ax.step(tt, S_, where="post", color=colors[gval], lw=1.2, label=lbl)
        ax.fill_between(tt, lo, hi, step="post", color=colors[gval], alpha=0.12, lw=0)
    ax.set_ylim(0, 1.02); ax.set_xlim(0, 15); ax.set_xlabel("Years"); ax.set_ylabel("Overall survival"); ax.set_title(title, fontsize=8)
    ax.text(0.97, 0.97, f"log-rank p = {p_:.2g}", transform=ax.transAxes, fontsize=7, ha="right", va="top")
    ax.legend(loc="lower left", fontsize=6.5)
ba.panel_label(axes[0], "a", x=-0.15); ba.panel_label(axes[1], "b", x=-0.15)
fig.tight_layout(); savefig(fig, "S2_Fig_survival", "S2_Fig")

In [ ]:
par_path = MATLAB_DIR / "ParetoSurface_100sol.csv"
if par_path.exists():
    par = pd.read_csv(par_path); par["sample"] = par.Model.map(to_sample_id)
    par = par.merge(df[["sample", "sample_type"]], on="sample")
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))
    for ax, yv, lbl in zip(axes, ["CU", "EA"], ["Carbon uptake (CU)", "Enzyme allocation (EA)"]):
        for t in ["Normal", "Tumor"]:
            g = par[par.sample_type == t]
            bins = pd.cut(g.Biomass / g.groupby("sample").Biomass.transform("max"), np.linspace(0, 1, 11))
            q = g.groupby(bins, observed=True)[yv].quantile([0.25, 0.5, 0.75]).unstack()
            xm = [b.mid for b in q.index]
            ax.plot(xm, q[0.5], color=COL[t], marker="o", ms=3, label=t); ax.fill_between(xm, q[0.25], q[0.75], color=COL[t], alpha=0.15, lw=0)
        ax.set_xlabel("Biomass (fraction of maximum)"); ax.set_ylabel(lbl); ax.legend()
    ba.panel_label(axes[0], "a", x=-0.2); ba.panel_label(axes[1], "b", x=-0.2)
    fig.tight_layout(); savefig(fig, "S3_Fig_pareto", "S3_Fig")

In [ ]:
NUM["run_at"] = datetime.datetime.now().isoformat(timespec="seconds")
json.dump(NUM, open(OUT_DIR / "paper_numbers.json", "w"), indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("done")